In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [ ]:
transactions = pd.read_csv("C:/Users/Admin/OneDrive - UWA/Projects/Data Engineering/Data-Engineering/Transaction Fraud Detection Project/Data/Raw/transactions.csv")
transactions.head()

Parse and convert *timestamp* column from string to datetime

In [ ]:
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

In [ ]:
transactions["log_amount"] = np.log1p(transactions["amount"])

In [ ]:
transactions["is_night"] = transactions["timestamp"].dt.hour.apply(lambda x: 1 if (x >= 0 and x < 5) else 0)

In [ ]:
digital_channels = ["mobile_app", "web_browser", "phone_ivr"]
transactions.loc[~transactions["device_type"].isin(digital_channels), "ip_risk_score"] = 0

In [ ]:
transactions["account_age_bucket"] = pd.cut(
    transactions["account_age_days"],
    bins = [0, 90, 180, 365, float("inf")],
    labels = ["new_90d", "new_180d", "new_1y", "established"],
    right = True
)

transactions["account_age_bucket_enc"] = LabelEncoder().fit_transform(transactions["account_age_bucket"].astype(str))

In [ ]:
transactions["is_unusual_spend"] = (transactions["amount_vs_avg_ratio"] > 5).astype(int)

In [ ]:
transactions["is_high_velocity"] = (transactions["velocity_1h"] > 3).astype(int)

In [ ]:
transactions["txn_to_limit_ratio"] = transactions["amount"] / transactions["credit_limit"]

Drop redundant columns *hour_of_day*, *day_of_week* as these data are all directly derivable from *timestamp* and *mcc_code* relates to *merchant_category*

Keeping these columns creates risks of inconsistency and bloats the feature matrix

In [ ]:
transactions.drop(columns = ["hour_of_day", "day_of_week", "mcc_code"], inplace = True)

One-hot encoding for Logistic regression

In [ ]:
transactions_linear = transactions.copy()
transactions_linear = pd.get_dummies(transactions_linear, 
                                     columns = ["merchant_category", "device_type"],
                                     drop_first = True)
# merchant_country have too many unique vlues for one-hot encoding, 
# so we label encode them for logistic regression
le = LabelEncoder()
transactions_linear["merchant_country_enc"] = le.fit_transform(transactions_linear["merchant_country"].astype(str))

Label encoding for XGBoost and LightBGM notebook

In [ ]:
for col in ["merchant_category", "merchant_country", "device_type"]:
    le = LabelEncoder()
    transactions[f"{col}_enc"] = le.fit_transform(transactions[col].astype(str))

In [ ]:
transactions.drop(columns = ["merchant_category", "merchant_country", "device_type"], inplace = True)

In [ ]:
account_dim = account_profiles[[
    "account_id",
    "home_country",
    "risk_score",
    "is_high_risk",
    "avg_monthly_txns",
    "pct_foreign",
    "avg_ip_risk",
    "unique_countries",
    "account_type"
]]

transactions = transactions.merge(account_dim, on = "account_id", how = "left")

In [ ]:
# Columns to exclude from model input
EXCLUDE = [
    "transaction_id", "account_id",          # identifiers
    "timestamp",                              # datetime, features already extracted
    "merchant_category", "merchant_country",  # raw strings, encoded versions exist
    "device_type", "account_age_bucket",      # raw strings, encoded versions exist
    "account_type", "home_country",           # raw strings, encoded versions exist
    "fraud_pattern",                          # target-related, not a feature
    "is_fraud",                               # target variable
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]
TARGET = "is_fraud"

print(f"Number of features: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")